## 코사인 유사도(Cosine Similarity)

코사인 유사도는 두 벡터 간의 코사인 각도를 이용하여 구할 수 있는 두 벡터의 유사도를 의미한다. 

두 벡터의 방향이 완전히 동일한 경우에는 1의 값을 가지며, 90°의 각을 이루면 0, 180°로 반대의 방향을 가지면 -1의 값을 갖게 된다.

즉, 두 벡터가 가리키는 방향이 얼마나 유사한가를 의미한다.

가령 아래와 같은 예시 문서들이 있다고 하자.

문서1 : 저는 사과 좋아요

문서2 : 저는 바나나 좋아요

문서3 : 저는 바나나 좋아요 저는 바나나 좋아요

이를 바탕으로 DTM을 만들면

In [9]:
import pandas as pd

docs = [
  '저는 사과 좋아요',
  '저는 바나나 좋아요',
  '저는 바나나 좋아요 저는 바나나 좋아요',
] 
# 1. 단어장 만들기 (sorted 사용)
vocab = sorted(list(set(w for doc in docs for w in doc.split())))

result = []
for i in range(len(docs)):
    result.append([])
    d = docs[i].split() # 문서를 단어 리스트로 분리
    for j in range(len(vocab)):
        t = vocab[j]
        # 리스트 내에서 정확히 그 단어와 일치하는 개수만 카운트
        result[-1].append(d.count(t))

tf_ = pd.DataFrame(result, columns=vocab)
tf_

,바나나,사과,저는,좋아요
0,0,1,1,1
1,1,0,1,1
2,2,0,2,2


In [2]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A, B):
    return dot(A, B)/(norm(A)*norm(B))

doc1 = np.array([0,1,1,1])
doc2 = np.array([1,0,1,1])
doc3 = np.array([2,0,2,2])

print(f'문서 1과 문서2의 유사도 : {cos_sim(doc1, doc2):.2f}')
print(f'문서 1과 문서3의 유사도 : {cos_sim(doc1, doc3):.2f}')
print(f'문서 2와 문서3의 유사도 : {cos_sim(doc2, doc3):.2f}')


문서 1과 문서2의 유사도 : 0.67
문서 1과 문서3의 유사도 : 0.67
문서 2와 문서3의 유사도 : 1.00


코사인 유사도는 유사도를 구할 때 벡터의 방향(패턴)에 초점을 두므로
 
문서의 길이가 다른 상황에서 비교적 공정한 비교를 할 수 있도록 도와준다.

### 코사인 유사도를 이용한 추천 시스템 구현하기

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

data = pd.read_csv(r'files/movies_metadata.csv', low_memory=False)

# 상위 2만개의 샘플을 data에 저장
data = data.head(20000)

# 결측값을 빈 값으로 대체
data['overview'] = data['overview'].fillna('')

tfidf = TfidfVectorizer(stop_words='english')   # tf-dif행렬 생성
tfidf_matrix = tfidf.fit_transform(data['overview'])
print('TF-IDF 행렬의 크기(shape) :',tfidf_matrix.shape)

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print('코사인 유사도 연산 결과 :',cosine_sim.shape)

title_to_index = dict(zip(data['title'], data.index))

# 영화 제목 Father of the Bride Part II의 인덱스를 리턴
idx = title_to_index['Father of the Bride Part II']
print(idx)


TF-IDF 행렬의 크기(shape) : (20000, 47487)
코사인 유사도 연산 결과 : (20000, 20000)
4


In [4]:
def get_recommendations(title, cosine_sim=cosine_sim):
    # 선택한 영화의 타이틀로부터 해당 영화의 인덱스를 받아온다.
    idx = title_to_index[title]

    # 해당 영화와 모든 영화와의 유사도를 가져온다.
    sim_scores = list(enumerate(cosine_sim[idx]))

    # 유사도에 따라 영화들을 정렬한다.
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # 가장 유사한 10개의 영화를 받아온다.
    sim_scores = sim_scores[1:11]

    # 가장 유사한 10개의 영화의 인덱스를 얻는다.
    movie_indices = [idx[0] for idx in sim_scores]

    # 가장 유사한 10개의 영화의 제목을 리턴한다.
    return data['title'].iloc[movie_indices]

In [5]:
get_recommendations('The Dark Knight Rises')

12481                            The Dark Knight
150                               Batman Forever
1328                              Batman Returns
15511                 Batman: Under the Red Hood
585                                       Batman
9230          Batman Beyond: Return of the Joker
18035                           Batman: Year One
19792    Batman: The Dark Knight Returns, Part 1
3095                Batman: Mask of the Phantasm
10122                              Batman Begins
Name: title, dtype: object

### 유클리드 거리를 통한 유사도 계산

In [1]:
import numpy as np

def dist(x,y):   
    return np.sqrt(np.sum((x-y)**2))

doc1 = np.array((2,3,0,1))
doc2 = np.array((1,2,3,1))
doc3 = np.array((2,1,2,2))
docQ = np.array((1,1,0,1))

print('문서1과 문서Q의 거리 :',dist(doc1,docQ))
print('문서2과 문서Q의 거리 :',dist(doc2,docQ))
print('문서3과 문서Q의 거리 :',dist(doc3,docQ))

문서1과 문서Q의 거리 : 2.23606797749979
문서2과 문서Q의 거리 : 3.1622776601683795
문서3과 문서Q의 거리 : 2.449489742783178


유클리드 거리의 값이 가장 작다는 것은 문서 간 거리가 가장 가깝다는 것을 의미한다. 

즉, 문서1이 문서Q와 가장 유사하다고 볼 수 있다.

### 자카드 유사도(Jaccard similarity)

A와 B 두개의 집합이 있다고 할 때 자카드 유사도 J는 아래 공식에 따라 계산한다.

$$ J(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|} $$

In [2]:
doc1 = "apple banana everyone like likey watch card holder"
doc2 = "apple banana coupon passport love you"

# 토큰화
tokenized_doc1 = doc1.split()
tokenized_doc2 = doc2.split()

print('문서1 :',tokenized_doc1)
print('문서2 :',tokenized_doc2)


문서1 : ['apple', 'banana', 'everyone', 'like', 'likey', 'watch', 'card', 'holder']
문서2 : ['apple', 'banana', 'coupon', 'passport', 'love', 'you']


In [4]:
union = set(tokenized_doc1).union(set(tokenized_doc2))
intersection = set(tokenized_doc1).intersection(set(tokenized_doc2))
print('문서1과 문서2의 합집합 :',union)
print('문서1과 문서2의 교집합 :',intersection)
print('자카드 유사도 :',len(intersection)/len(union))


문서1과 문서2의 합집합 : {'card', 'apple', 'passport', 'you', 'coupon', 'watch', 'like', 'likey', 'banana', 'love', 'everyone', 'holder'}
문서1과 문서2의 교집합 : {'banana', 'apple'}
자카드 유사도 : 0.16666666666666666


텍스트 임베딩

[TextEmbedding](TextEmbedding.ipynb)